# Supervised Learning: Identifying Crypto Mining Data Centers

The FracTracker data center list does not say which facilities are crypto mines. The separate FracTracker crypto tracker does, but the two files do not share an ID.

This notebook starts by joining them on location (`lat`, `long`): if a data center sits close enough to a known crypto site, we label it as crypto.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from geopy.distance import geodesic

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load data

The data centers come from `02-data-cleaning.ipynb`.

The crypto file has not been cleaned yet, so it still comes from `data/raw/`.

In [ ]:
data_centers = pd.read_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv")
crypto = pd.read_csv(RAW_DIR / "fractracker_crypto.csv")

print("Data centers:", data_centers.shape)
print("Crypto sites:", crypto.shape)

### Drop invalid coordinates

A few rows have coordinates that cannot be real: a longitude of `-11402.57827` (likely a typo) and a data center at `0, 0` (missing location). We can't measure distance for these, so we drop them.

In [ ]:
def valid_coordinates(df):
    return df["lat"].between(-90, 90) & df["long"].between(-180, 180) & ~((df["lat"] == 0) & (df["long"] == 0))

print("Dropping data centers:", data_centers.loc[~valid_coordinates(data_centers), "facility_name"].tolist())
print("Dropping crypto sites:", crypto.loc[~valid_coordinates(crypto), "Name"].tolist())

data_centers = data_centers[valid_coordinates(data_centers)].reset_index(drop=True)
crypto = crypto[valid_coordinates(crypto)].reset_index(drop=True)

## Distance between every crypto site and every data center

`geodesic` from the `geopy` package takes two `(lat, long)` points and returns the distance between them along the Earth's surface.

We loop over every crypto site and every data center, which is about 500,000 pairs, so this cell takes around 30 seconds.

The result is a table with one row per crypto site and one column per data center.

In [ ]:
# Coordinates format: (latitude, longitude)
crypto_points = list(zip(crypto["lat"], crypto["long"]))
dc_points = list(zip(data_centers["lat"], data_centers["long"]))

distances = []
for crypto_point in crypto_points:
    row = []
    for dc_point in dc_points:
        row.append(geodesic(crypto_point, dc_point).miles)
    distances.append(row)

# turn the list of rows into a table numpy can search
distances = np.array(distances)

print("Distance table (crypto sites, data centers):", distances.shape)

## Nearest data center for each crypto site

Before picking a cutoff, look at how far each crypto site is from its closest data center.

In [ ]:
nearest_dc = distances.argmin(axis=1)

crypto["nearest_dc_miles"] = distances.min(axis=1)
crypto["nearest_dc_name"] = data_centers["facility_name"].values[nearest_dc]
crypto["nearest_dc_operator"] = data_centers["operator_name"].values[nearest_dc]

for miles in [0.1, 0.25, 0.5, 1, 2, 5]:
    # keep only the crypto sites that are close enough, then count them
    nearby = crypto[crypto["nearest_dc_miles"] <= miles]
    print(f"Crypto sites with a data center within {miles} mi: {len(nearby)}")

Most crypto sites have no data center nearby. Many crypto mines are simply not in the data center list.

The table below puts the names side by side so we can check whether close pairs are really the same facility.

In [ ]:
columns = [
    "Name", "Company", "Precise Location?",
    "nearest_dc_name", "nearest_dc_operator", "nearest_dc_miles"
]

crypto.loc[crypto["nearest_dc_miles"] <= 2, columns].sort_values("nearest_dc_miles")

## Label the data centers

We label from the data center side: for each data center, find the closest crypto site. If it is within `MAX_MILES`, the data center gets `is_crypto = 1`.

Labeling this way handles two things:
- The crypto file repeats some sites (same name or same coordinates), and several crypto rows can point to one data center.
- Every data center gets exactly one label.

`0.5` miles is a starting point. Some sites under 2 miles are clearly different facilities, for example CleanSpark (Clinton) and Amazon Clinton Data Center.

In [ ]:
MAX_MILES = 0.5


nearest_crypto = distances.argmin(axis=0)

data_centers["crypto_miles"] = distances.min(axis=0)
data_centers["crypto_name"] = crypto["Name"].values[nearest_crypto]
data_centers["crypto_company"] = crypto["Company"].values[nearest_crypto]
data_centers["crypto_status"] = crypto["Status"].values[nearest_crypto]

data_centers["is_crypto"] = (data_centers["crypto_miles"] <= MAX_MILES).astype(int)

# only keep crypto details for data centers that matched
not_matched = data_centers["is_crypto"] == 0
data_centers.loc[not_matched, ["crypto_name", "crypto_company", "crypto_status"]] = np.nan

print("Data centers labeled crypto:", data_centers["is_crypto"].sum(), "of", len(data_centers))
print("Share labeled crypto:", round(data_centers["is_crypto"].mean() * 100, 1), "%")

In [ ]:
columns = [
    "facility_name", "operator_name", "state", "status",
    "crypto_name", "crypto_company", "crypto_miles"
]

data_centers.loc[data_centers["is_crypto"] == 1, columns].sort_values("crypto_miles")

## How much does the cutoff matter?

The number of positive labels depends on `MAX_MILES`. This helps us choose it.

In [ ]:
rows = []
for miles in [0.1, 0.25, 0.5, 1, 2]:
    labeled_dcs = data_centers[data_centers["crypto_miles"] <= miles]
    matched_crypto = crypto[crypto["nearest_dc_miles"] <= miles]

    rows.append({
        "max_miles": miles,
        "data_centers_labeled": len(labeled_dcs),
        "crypto_sites_matched": len(matched_crypto),
    })

pd.DataFrame(rows)

# Part 2: Train a classifier

Now that every data center has an `is_crypto` label, we can train a model to predict it.

Following the steps from SIADS 542:
1. Pick the features (`X`) and the label (`y`).
2. Split into a training set and a test set.
3. Get a baseline score from a dummy classifier.
4. Train a logistic regression and compare it to the baseline.
5. Look at what the model learned and where it was wrong.

## Pick the features

We kept the features simple:
- `status`: Operating, Proposed, Cancelled, ...
- `sizerank`: the facility's size bucket, from Small to Mega campus, or Unknown.
- `has_sqft`: 1 if FracTracker lists the building's square footage, 0 if not.

`status` and `sizerank` are categories, so we one-hot encode them with `pd.get_dummies`. Each category becomes its own binary varable.

Columns we leave out on purpose to prevent data leakage:
- `purpose`: a few rows literally say "Crypto". That gives away the answer (data leakage).
- `lat` and `long`: we used location to make the label, so the model shouldn't use it to predict the label.
- `state`: one-hot encoding would create about 50 extra columns, too many for the dataset with only 41 crypto data centers.

In [ ]:
# Another data preprocessing step we might move earlier in the pipeline
# fix Mega Campus category that is spelled two ways
data_centers["sizerank"] = data_centers["sizerank"].replace(
    "Mega Campus (>1,000 MW)", "Mega campus (>1,000 MW)"
)

X = pd.get_dummies(data_centers[["status", "sizerank"]], dtype=int)
X["has_sqft"] = data_centers["facility_size_sqft"].notna().astype(int)

y = data_centers["is_crypto"]

print("X:", X.shape)
print("y:", y.shape)
X.head()

## Split into training and test sets

`train_test_split` holds back 25% of the rows as a test set. The model never sees these rows during training.

Crypto data centers are rare, only about 2%. Using `stratify=y` keeps the same proportion of crypto rows in both sets.

We set `random_state=0` for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

print("Training rows:", len(X_train), "| crypto:", y_train.sum())
print("Test rows:", len(X_test), "| crypto:", y_test.sum())

## Baseline: a dummy classifier

We set up a dummy classifier as a baseline with `strategy="most_frequent"`. This way all our data gets the same prediction, 'not crypto'.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("Dummy accuracy:", round(dummy.score(X_test, y_test), 3))

The dummy is about 98% accurate without finding a single crypto data center.

(This might be a good point to talk about accuracy as a metric that can be misleading when classes are imbalanced. Maybe introduce precision...)

## Logistic regression: first try

Logistic regression adds up the features, each multiplied by a weight (coefficient), and turns the total into a probability between 0 and 1. If the probability is 0.5 or more, it predicts crypto.

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(model.score(X_test, y_test), 3))

# rows = actual (0, 1), columns = predicted (0, 1)
print(confusion_matrix(y_test, predictions))

The model predicts "not crypto" for every row. With 98% of the training rows labeled 0, always guessing 0 is the easiest way for the model to be right most of the time.

## Logistic regression: balanced class weights

Changing tactics, setting `class_weight="balanced"` sets a risk/reward the weights a mistake on a crypto label more than a mistake on a non-crypto row. Given how rare crypto is in the dataset. Now ignoring the crypto rows is expensive, so the model has to try to find them.

In [ ]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(model.score(X_test, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

Accuracy went down, but the model now finds some of the crypto data centers.

In [ ]:
coefficients = pd.Series(model.coef_[0], index=X.columns)
coefficients.sort_values()